# Triple-Barrier Parameter Analysis\n\nThis notebook provides a workflow for analyzing and visualizing the choice of parameters for the Triple-Barrier labeling method.\n\nThe process is as follows:\n1. **Load Execution Price Data**: Fetch the realistic bid/ask execution prices for a given market and date range.\n2. **Calculate & Visualize Volatility**: Compute a rolling volatility measure from the historical data, which will be used to set dynamic barriers.\n3. **Define Barriers & Generate Labels**: Use the calculated volatility to define the triple barriers and generate the resulting labels.\n4. **Analyze Label Distribution**: Summarize and plot the distribution of outcomes (Win, Loss, Timeout) to understand the impact of the chosen parameters.

## Setup\n\nImport libraries and define the core parameters for the analysis.

In [ ]:
import sys\nsys.path.insert(0, '../src')\n\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom qate.core.model import Market, Exchange, Symbol\nfrom qate.store.parquet_store import ParquetStore\nfrom qate.util.dt_range import DtRange\n\n# --- Display and Plotting Settings ---\npd.set_option('display.max_columns', 50)\npd.set_option('display.max_rows', 100)\nsns.set_style(\"whitegrid\")\n%matplotlib inline\n\n# --- Core Analysis Parameters ---\nPARQUET_STORE_ROOT_DIR = \"../data/parquet\" # Adjust this path as needed\nMARKET = Market(Exchange.BINANCE, Symbol.BTC_USDT) # Example market\nEXECUTION_SIZE = \"1.0\" # The order size for which to get execution prices\n\n# --- Date Range for Analysis ---\nDATA_START_DATE = \"2023-01-03\"\nDATA_END_DATE = \"2023-01-04\" # Analyze a single day of data\nDATA_DT_RANGE = DtRange(DATA_START_DATE, DATA_END_DATE)\n\n# --- Triple-Barrier Hyperparameters ---\nVOLATILITY_LOOKBACK_MINUTES = 60 # Use a 1-hour window for rolling volatility\nBARRIER_MULTIPLIER = 2.0 # Set barriers at 2.0x the rolling volatility\nTIMEOUT_SEC = 45 # Set a 45-second timeout for the vertical barrier\nTIMEOUT_DELTA = pd.Timedelta(seconds=TIMEOUT_SEC)\n\nprint(\"Setup complete. Parameters are defined.\")

## 1. Load Execution Price Data\n\nLoad the execution price data which contains the realistic bid, ask, and mid prices for our chosen market and order size.

In [ ]:
full_parquet_store = ParquetStore(PARQUET_STORE_ROOT_DIR)\n\n# We need to read a slightly larger range to ensure the volatility window is populated\nvol_read_start_dt = pd.to_datetime(DATA_START_DATE) - pd.Timedelta(minutes=VOLATILITY_LOOKBACK_MINUTES)\nload_range = DtRange(vol_read_start_dt.strftime('%Y-%m-%d %H:%M:%S'), DATA_END_DATE)\n\nprint(f\"Loading execution prices for {MARKET.symbol.name} (size: {EXECUTION_SIZE})...\")\nprice_store = full_parquet_store.create_execution_price(MARKET, execution_size_str=EXECUTION_SIZE)\ndf = price_store.read(load_range)\n\nif 'ts' in df.columns:\n    df.index = pd.to_datetime(df['ts'], unit='s')\n    df = df.drop(columns=['ts'])\n\nprint(f\"Loaded {len(df)} data points from {df.index.min()} to {df.index.max()}.\")\ndf.head()

## 2. Calculate & Visualize Volatility\n\nHere we implement **Methodology 1**: calculating dynamic barriers based on recent volatility. We'll compute the rolling standard deviation of 1-minute returns.

In [ ]:
# 1. Resample mid-price to 1-minute bars to calculate returns\none_min_mid_price = df['mid'].resample('1min').last()\n\n# 2. Calculate 1-minute percentage returns\none_min_returns = one_min_mid_price.pct_change()\n\n# 3. Calculate the rolling standard deviation (volatility)\n# This gives us a volatility estimate at a 1-minute frequency\nrolling_vol_1min = one_min_returns.rolling(window=VOLATILITY_LOOKBACK_MINUTES).std()\n\n# 4. Align the volatility back to the original tick-level DataFrame\n# We forward-fill to apply the last known volatility to each tick within that minute.\ndf['volatility'] = rolling_vol_1min.reindex(df.index, method='ffill')\n\n# Clean up data to be within the original desired date range\ndf = df.loc[DATA_DT_RANGE.start_dt:DATA_DT_RANGE.end_dt].dropna()\n\nprint(f\"Calculated rolling volatility. {len(df)} data points remaining after alignment.\")\ndf[['mid', 'volatility']].head()

In [ ]:
# Plot the mid-price and the calculated volatility\nfig, ax1 = plt.subplots(figsize=(15, 7))\n\ncolor = 'tab:blue'\nax1.set_xlabel('Time')\nax1.set_ylabel('Mid Price', color=color)\nax1.plot(df.index, df['mid'], color=color, label='Mid Price')\nax1.tick_params(axis='y', labelcolor=color)\n\nax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis\ncolor = 'tab:red'\nax2.set_ylabel('1-Hour Rolling Volatility', color=color)\nax2.plot(df.index, df['volatility'], color=color, alpha=0.7, label='Volatility')\nax2.tick_params(axis='y', labelcolor=color)\n\nplt.title(f'{MARKET.symbol.name} Mid Price and Rolling Volatility')\nfig.tight_layout()\nplt.show()

## 3. Define Barriers & Generate Labels\n\nNow we use the calculated `volatility` series to define our dynamic barriers and generate labels. For simplicity, we will only generate labels for a **long scenario**.

In [ ]:
# Define the profit-take and stop-loss levels for each timestamp dynamically\ndf['pt_pct'] = BARRIER_MULTIPLIER * df['volatility']\ndf['sl_pct'] = BARRIER_MULTIPLIER * df['volatility']\n\nprint(\"Profit-take and Stop-loss levels (in %):\")\n(df[['pt_pct', 'sl_pct']] * 100).describe()

In [ ]:
def get_labels_with_dynamic_barriers(prices_df: pd.DataFrame, timeout_delta: pd.Timedelta) -> pd.DataFrame:\n    \"\"\"\n    Computes triple-barrier labels for a long scenario using dynamic pt/sl levels.\n    `prices_df` must contain 'ask', 'bid', 'pt_pct', 'sl_pct' columns.\n    \"\"\"\n    n = len(prices_df)\n    results = []\n    \n    # Extract series for faster access\n    entry_ask_prices = prices_df['ask']\n    future_bid_prices = prices_df['bid']\n    pt_pct_series = prices_df['pt_pct']\n    sl_pct_series = prices_df['sl_pct']\n\n    for i in range(n):\n        entry_ts = prices_df.index[i]\n        entry_price = entry_ask_prices.iloc[i]\n        timeout_ts = entry_ts + timeout_delta\n\n        # Get the specific pt/sl for this entry time\n        pt_level = pt_pct_series.iloc[i]\n        sl_level = sl_pct_series.iloc[i]\n\n        if pd.isna(entry_price) or pd.isna(pt_level) or pd.isna(sl_level):\n            continue\n\n        upper_barrier = entry_price * (1 + pt_level)\n        lower_barrier = entry_price * (1 - sl_level)\n\n        # Find future path\n        future_path = future_bid_prices.loc[future_bid_prices.index > entry_ts]\n        \n        # Find barrier hits\n        pt_hits = future_path[future_path >= upper_barrier]\n        sl_hits = future_path[future_path <= lower_barrier]\n\n        first_pt_hit_ts = pt_hits.index[0] if not pt_hits.empty else None\n        first_sl_hit_ts = sl_hits.index[0] if not sl_hits.empty else None\n        \n        first_hit_ts = min(ts for ts in [first_pt_hit_ts, first_sl_hit_ts] if ts) if any([first_pt_hit_ts, first_sl_hit_ts]) else None\n        \n        label = 0\n        if first_hit_ts and first_hit_ts <= timeout_ts:\n            if future_path[first_hit_ts] >= upper_barrier:\n                label = 1 # Profit Take\n            else:\n                label = -1 # Stop Loss\n        else:\n            label = 0 # Timeout\n\n        results.append({\"label\": label})\n\n    return pd.DataFrame(results, index=prices_df.index[:len(results)])\n\n# Generate the labels\nprint(\"Generating labels... (This may take a few moments)\")\nlabels_df = get_labels_with_dynamic_barriers(df, TIMEOUT_DELTA)\ndf = df.join(labels_df)\nprint(\"Label generation complete.\")\ndf[['mid', 'volatility', 'pt_pct', 'sl_pct', 'label']].head()

## 4. Analyze Label Distribution\n\nFinally, we analyze the outcomes. How many of our potential trades resulted in a win, a loss, or a timeout? This is critical for understanding if the barrier parameters are well-chosen. If 99% of labels are timeouts, the barriers might be too wide or the timeout too short.

In [ ]:
label_counts = df['label'].value_counts().sort_index()\n\n# Create a mapping for human-readable labels\nlabel_map = {\n    1: f'Win (PT = {BARRIER_MULTIPLIER}x Vol)',\n    -1: f'Loss (SL = {BARRIER_MULTIPLIER}x Vol)',\n    0: f'Timeout ({TIMEOUT_SEC}s)'\n}\nlabel_counts.rename(index=label_map, inplace=True)\n\nprint(\"Label Distribution Summary:\")\nsummary_df = pd.DataFrame({\n    'count': label_counts,\n    'percentage': (label_counts / label_counts.sum() * 100).round(2)\n})\nprint(summary_df)\n\n# Plot the distribution\nplt.figure(figsize=(10, 6))\nax = sns.barplot(x=summary_df.index, y=summary_df['count'])\nax.set_title('Triple-Barrier Label Distribution')\nax.set_ylabel('Count')\nax.set_xlabel('Outcome')\n\n# Add percentage labels to the bars\nfor i, p in enumerate(ax.patches):\n    height = p.get_height()\n    ax.text(p.get_x() + p.get_width()/2.,\n            height + 0.1,\n            f\"{summary_df['percentage'].iloc[i]}%\",\n            ha=\"center\", va=\"bottom\")\n\nplt.xticks(rotation=15)\nplt.tight_layout()\nplt.show()